In [0]:
%run ../../commonsetup/Setup-Common


### Gold: real-time vehicle positions + trip delays, enriched with static GTFS dimensions
##### Joins the two streaming Silver facts (vehicle_positions, trip_updates) against the conformed static dims (routes/trips/stops).
##### Stream-static join: the dim side is read as a batch DataFrame and broadcast, refreshed each run (same trigger(availableNow=True) pattern as the rest of the pipeline), so it's no staler than anything else here.
##### Route resolved as coalesce(RT feed's own route_id, static trip's route_id) -- NonTimetabled/UNSCHEDULED trips carry neither, which is expected, not a bug. Every join is LEFT so an unresolved trip still produces a row (route/stop names NULL) instead of disappearing.

In [0]:
from pyspark.sql import functions as F

In [0]:
# ---- Config ---------------------------------------------------------------
VP_SOURCE_TABLE       = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_vehicle_positions"
TU_SOURCE_TABLE       = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_trip_updates"
ROUTES_TABLE          = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_routes"
TRIPS_TABLE           = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_trips"
STOPS_TABLE           = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_stops"

GOLD_VP_TABLE         = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_vehicle_positions_enriched"
GOLD_TU_TABLE         = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_delays_enriched"

CHECKPOINT_GOLD_VP    = f"{CHECKPOINT_BASE}/gold_vehicle_positions_enriched/checkpoint"
CHECKPOINT_GOLD_TU    = f"{CHECKPOINT_BASE}/gold_trip_delays_enriched/checkpoint"

### 1: Load static dimensions (batch, broadcast)
##### Narrowed to just the columns Gold needs -- a broadcast join ships this to every executor, so thin matters more here than in the dims themselves.

In [0]:
routes_dim = (
    spark.table(ROUTES_TABLE)
    .select("gtfs_mode", "route_id", "route_short_name", "route_long_name")
)

trips_dim = (
    spark.table(TRIPS_TABLE)
    .select(
        "gtfs_mode", "trip_id",
        F.col("route_id").alias("trip_route_id"),
        "trip_headsign", "direction_id",
    )
)

stops_dim = (
    spark.table(STOPS_TABLE)
    .select("gtfs_mode", "stop_id", "stop_name")
)

### 2: Gold -- vehicle positions enriched
##### Same grain as Silver, purely additive columns (trip_headsign, route names, stop name) via LEFT join.

In [0]:
vp_stream = spark.readStream.table(VP_SOURCE_TABLE)

vp_enriched = (
    vp_stream.alias("vp")
    .join(F.broadcast(trips_dim).alias("t"),
          on=[F.col("vp.gtfs_mode") == F.col("t.gtfs_mode"),
              F.col("vp.trip_id") == F.col("t.trip_id")],
          how="left")
    .withColumn("resolved_route_id", F.coalesce(F.col("vp.route_id"), F.col("t.trip_route_id")))
    .join(F.broadcast(routes_dim).alias("r"),
          on=[F.col("vp.gtfs_mode") == F.col("r.gtfs_mode"),
              F.col("resolved_route_id") == F.col("r.route_id")],
          how="left")
    .join(F.broadcast(stops_dim).alias("s"),
          on=[F.col("vp.gtfs_mode") == F.col("s.gtfs_mode"),
              F.col("vp.stop_id") == F.col("s.stop_id")],
          how="left")
    .select(
        F.col("vp.entity_id"),
        F.col("vp.gtfs_mode"),
        F.col("vp.vehicle_id"),
        F.col("vp.vehicle_label"),
        F.col("vp.trip_id"),
        F.col("resolved_route_id").alias("route_id"),
        F.col("r.route_short_name"),
        F.col("r.route_long_name"),
        F.col("t.trip_headsign"),
        F.col("t.direction_id"),
        F.col("vp.schedule_relationship"),
        F.col("vp.latitude"),
        F.col("vp.longitude"),
        F.col("vp.position_timestamp"),
        F.col("vp.stop_id"),
        F.col("s.stop_name"),
        F.col("vp.congestion_level"),
        F.coalesce(F.col("vp.occupancy_status"), F.lit("Not Reported")).alias("occupancy_status"),
        F.col("vp.poll_timestamp"),
        F.col("vp.ingestion_timestamp"),
    )
)

vp_gold_query = (
    vp_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_GOLD_VP)
    .partitionBy("gtfs_mode")
    .trigger(availableNow=True)
    .toTable(GOLD_VP_TABLE)
)

### 3: Gold -- trip delays enriched
##### Same grain as Silver trip_updates. Delay fields are TfNSW's own reported values (GTFS-RT StopTimeUpdate.delay) -- Gold makes them legible (route, stop, direction), doesn't recompute them.

In [0]:
tu_stream = spark.readStream.table(TU_SOURCE_TABLE)

tu_enriched = (
    tu_stream.alias("tu")
    .join(F.broadcast(trips_dim).alias("t"),
          on=[F.col("tu.gtfs_mode") == F.col("t.gtfs_mode"),
              F.col("tu.trip_id") == F.col("t.trip_id")],
          how="left")
    .withColumn("resolved_route_id", F.coalesce(F.col("tu.route_id"), F.col("t.trip_route_id")))
    .join(F.broadcast(routes_dim).alias("r"),
          on=[F.col("tu.gtfs_mode") == F.col("r.gtfs_mode"),
              F.col("resolved_route_id") == F.col("r.route_id")],
          how="left")
    .join(F.broadcast(stops_dim).alias("s"),
          on=[F.col("tu.gtfs_mode") == F.col("s.gtfs_mode"),
              F.col("tu.stop_id") == F.col("s.stop_id")],
          how="left")
    .select(
        F.col("tu.gtfs_mode"),
        F.col("tu.trip_id"),
        F.col("resolved_route_id").alias("route_id"),
        F.col("r.route_short_name"),
        F.col("r.route_long_name"),
        F.col("t.trip_headsign"),
        F.col("t.direction_id"),
        F.col("tu.vehicle_id"),
        F.col("tu.start_date"),
        F.col("tu.stop_sequence"),
        F.col("tu.stop_id"),
        F.col("s.stop_name"),
        F.col("tu.arrival_delay_seconds"),
        F.col("tu.departure_delay_seconds"),
        F.col("tu.poll_timestamp"),
        F.col("tu.ingestion_timestamp"),
    )
)

tu_gold_query = (
    tu_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_GOLD_TU)
    .partitionBy("gtfs_mode")
    .trigger(availableNow=True)
    .toTable(GOLD_TU_TABLE)
)

vp_gold_query.awaitTermination()
tu_gold_query.awaitTermination()

print(f"gold_vehicle_positions_enriched: {spark.table(GOLD_VP_TABLE).count()} total rows")
print(f"gold_trip_delays_enriched: {spark.table(GOLD_TU_TABLE).count()} total rows")

### 4: Power BI-facing summary view
##### A VIEW, not a table -- it's a small aggregate over data already in Gold, so no separate streaming job to maintain, and it's current the moment new rows land. Power BI points at this directly, or at the two Gold tables for row-level detail.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_route_delay_summary_v AS
    SELECT
      gtfs_mode,
      route_id,
      COALESCE(route_short_name, 'Unscheduled/Unresolved') AS route_short_name,
      route_long_name,
      date_trunc('HOUR', poll_timestamp) AS poll_hour,
      COUNT(*) AS reading_count,
      ROUND(AVG(arrival_delay_seconds) / 60.0, 1) AS avg_arrival_delay_minutes,
      ROUND(MAX(arrival_delay_seconds) / 60.0, 1) AS max_arrival_delay_minutes,
      ROUND(AVG(departure_delay_seconds) / 60.0, 1) AS avg_departure_delay_minutes
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_delays_enriched
    WHERE arrival_delay_seconds IS NOT NULL OR departure_delay_seconds IS NOT NULL
    GROUP BY ALL
    ORDER BY poll_hour DESC, avg_arrival_delay_minutes DESC
""")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_vehicle_positions_current_v AS
    SELECT
      vp.gtfs_mode,
      vp.vehicle_id,
      vp.vehicle_label,
      vp.trip_id,
      COALESCE(vp.route_id, t.route_id) AS route_id,
      COALESCE(r.route_short_name, 'Unscheduled/Unresolved') AS route_short_name,
      r.route_long_name,
      t.trip_headsign, t.direction_id,
      vp.schedule_relationship, vp.latitude, vp.longitude, vp.position_timestamp,
      vp.stop_id, s.stop_name, vp.congestion_level,
      COALESCE(vp.occupancy_status, 'Not Reported') AS occupancy_status,
      vp.poll_timestamp
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_vehicle_positions_current vp
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_trips  t ON vp.gtfs_mode = t.gtfs_mode AND vp.trip_id = t.trip_id
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_routes r ON vp.gtfs_mode = r.gtfs_mode AND COALESCE(vp.route_id, t.route_id) = r.route_id
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_stops  s ON vp.gtfs_mode = s.gtfs_mode AND vp.stop_id = s.stop_id
""")

In [0]:
%sql
select * from gold_route_delay_summary_v limit 20;
